In [2]:
import requests

def get_kiel_bounding_box(factor=1):
    # Fetches Kiel's bounding box from Nominatim and expands it by the given factor.
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": "Kiel, Germany",
        "format": "json",
        "limit": 1
    }
    headers = {"User-Agent": "Data Science Project CAU WS25/26 Group 16"}

    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    bb = data[0]["boundingbox"]
    lat_min = float(bb[0])
    lat_max = float(bb[1])
    lon_min = float(bb[2])
    lon_max = float(bb[3])

    # Increase boundingbox size by "factor"
    lat_center = (lat_min + lat_max) / 2
    lon_center = (lon_min + lon_max) / 2
    lat_span   = (lat_max - lat_min) / 2
    lon_span   = (lon_max - lon_min) / 2

    return {
        "lat_min": lat_center - lat_span * factor,
        "lat_max": lat_center + lat_span * factor,
        "lon_min": lon_center - lon_span * factor,
        "lon_max": lon_center + lon_span * factor,
    }


def is_near_kiel(lat, lon, bb):
    return bb["lat_min"] <= lat <= bb["lat_max"] and bb["lon_min"] <= lon <= bb["lon_max"]

In [5]:
import os
import polars as pl
from pyproj import Transformer
from collections import defaultdict

BASE_DIR = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "BASt", "AggregatedDataForKiel")
YEARS    = [2021, 2022, 2023, 2024, 2025]
MONTHS   = [f"{m:02d}" for m in range(1, 13)]
VARIANTS = ["A", "B"]

transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=False)

def utm32_to_latlon(easting, northing):
    return transformer.transform(
        float(str(easting).replace(".", "").replace(",", ".")),
        float(str(northing).replace(".", "").replace(",", ".")),
    )


def get_kiel_stations(meta_file, bb):
    # Returns a list of station numbers located near Kiel.
    meta = pl.read_csv(meta_file, infer_schema_length=0)
    kiel_stations = []
    for row in meta.iter_rows(named=True):
        try:
            lat, lon = utm32_to_latlon(row["Koordinaten_UTM32_E"], row["Koordinaten_UTM32_N"])
            if is_near_kiel(lat, lon, bb):
                kiel_stations.append(int(row["Dauerzaehlstellennummer"]))
        except Exception:
            pass
    return kiel_stations


def main():
    bb = get_kiel_bounding_box(factor=3)
    station_months = defaultdict(set)
    filtered_data  = []

    for year in YEARS:
        for variant in VARIANTS:
            folder = os.path.join(BASE_DIR, "BASt", "AggregatedDataForSH", f"Data {year}", f"SH_{year}_{variant}_S")

            for month in MONTHS:
                meta_file = os.path.join(folder, f"Meta_{year}_{month}_{variant}_S.csv")
                roh_file  = os.path.join(folder, f"Roh_{year}_{month}_{variant}_S.csv")

                if not os.path.exists(meta_file) or not os.path.exists(roh_file):
                    continue

                kiel_stations = get_kiel_stations(meta_file, bb)

                for nr in kiel_stations:
                    station_months[nr].add(f"{year}-{month}")

                if kiel_stations:
                    roh = (
                        pl.read_csv(roh_file, infer_schema_length=0)
                        .with_columns(pl.col("Zst").str.strip_chars().cast(pl.Int64, strict=False).alias("Zst_int"))
                        .filter(pl.col("Zst_int").is_in(kiel_stations))
                        .drop("Zst_int")
                        .with_columns(pl.lit(year).alias("year"), pl.lit(month).alias("month"), pl.lit(variant).alias("variant"))
                    )
                    filtered_data.append(roh)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Save filtered raw data
    pl.concat(filtered_data, how="diagonal").write_csv(os.path.join(OUTPUT_DIR, "kiel_stations_raw.csv"))

    # Save station presence overview
    pl.DataFrame([
        {"station": nr, "months_present": len(monate), "months": ", ".join(sorted(monate))}
        for nr, monate in sorted(station_months.items())
    ]).write_csv(os.path.join(OUTPUT_DIR, "kiel_station_count.csv"))

    return station_months


if __name__ == "__main__":
    station_months = main()